# Analysing Openproblems Results

In [4]:
import os
import yaml
import pandas as pd

base_dir = "/Users/seohyon/resources/results"

rows = []

# Step 1 — Collect all rows
for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    yaml_path = os.path.join(folder_path, "score_uns.yaml")
    
    if os.path.isdir(folder_path) and os.path.exists(yaml_path):
        with open(yaml_path, "r") as f:
            try:
                data = yaml.safe_load(f)
            except yaml.YAMLError as e:
                print(f"⚠️ Could not read {yaml_path}: {e}")
                continue

        for entry in data:
            metrics = entry.get("metric_ids", [])
            values = entry.get("metric_values", [])
            
            for m_id, m_val in zip(metrics, values):
                rows.append({
                    "dataset_id": entry.get("dataset_id"),
                    "file_size": entry.get("file_size"),
                    "method_id": entry.get("method_id"),
                    "normalization_id": entry.get("normalization_id"),
                    "metric_id": m_id,
                    "metric_value": m_val
                })

# Step 2 — Create one big DataFrame
df = pd.DataFrame(rows)

# Step 3 — Split into separate tables per dataset_id
datasets = {}
for dataset_id, sub_df in df.groupby("dataset_id"):
    datasets[dataset_id] = sub_df.reset_index(drop=True)
    print(f"\n=== Dataset: {dataset_id} ===")
    print(sub_df)

    # Optional: Save each dataset table as its own CSV
    safe_name = dataset_id.replace("/", "_")  # clean file name
    output_path = f"/Users/seohyon/resources/{safe_name}_scores.csv"
    sub_df.to_csv(output_path, index=False)
    print(f"✅ Saved to {output_path}")




=== Dataset: cellxgene_census/dkd ===
                 dataset_id  file_size                         method_id  \
1      cellxgene_census/dkd      22632                    no_integration   
2      cellxgene_census/dkd      22632                    no_integration   
4      cellxgene_census/dkd      22632              no_integration_batch   
20     cellxgene_census/dkd      22632                         harmonypy   
48     cellxgene_census/dkd      22632  shuffle_integration_by_cell_type   
...                     ...        ...                               ...   
18912  cellxgene_census/dkd      22632                             mnnpy   
18916  cellxgene_census/dkd      22632                  embed_cell_types   
18918  cellxgene_census/dkd      22632                      scgpt_mlflow   
18926  cellxgene_census/dkd      22632                               uce   
18927  cellxgene_census/dkd      22632                              scvi   

      normalization_id                metric_id 